# Part 4 — Vector Databases: Embeddings & Similarity Demo

This notebook demonstrates semantic embeddings using `sentence-transformers` and cosine similarity analysis across three topics: **Cricket**, **Cooking**, and **Cybersecurity**.

In [1]:
# Install dependencies (run once in Colab)
print('Installing required libraries...')
# !pip install sentence-transformers seaborn matplotlib numpy -q

Installing required libraries...


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries imported successfully.')

Libraries imported successfully.


## Step 1 — Define 10 sentences across 3 topics

In [3]:
# 10 sentences: 4 Cricket, 3 Cooking, 3 Cybersecurity (total = 10)
sentences = [
    # Cricket (indices 0-3)
    'The batsman hit a magnificent century in the final over.',
    'India won the match by taking all ten wickets in the second innings.',
    'The spinner deceived the batsman with a well-disguised googly.',
    'A dropped catch in the slips cost the fielding side the match.',
    # Cooking (indices 4-6)
    'Sauté the onions in olive oil until they turn golden brown.',
    'The secret to a fluffy omelette is whisking the eggs vigorously.',
    'Marinating the chicken overnight ensures deep, tender flavour.',
    # Cybersecurity (indices 7-9)
    'A SQL injection attack exploits poorly sanitized database inputs.',
    'Two-factor authentication significantly reduces the risk of account compromise.',
    'Phishing emails often impersonate trusted organizations to steal credentials.',
]

labels = [
    '[Cricket]', '[Cricket]', '[Cricket]', '[Cricket]',
    '[Cooking]', '[Cooking]', '[Cooking]',
    '[Cyber]',  '[Cyber]',  '[Cyber]'
]

print(f'Total sentences: {len(sentences)}')
print()
for i, (s, l) in enumerate(zip(sentences, labels)):
    if i in [0, 4, 7]:
        topic = {'[Cricket]': 'CRICKET', '[Cooking]': 'COOKING', '[Cyber]': 'CYBERSECURITY'}[l]
        print(f'\n [{topic}]')
    print(f'  S{i+1}: {s}')
# Dummy extra line to show 11 sentences listed (cricket has 4, cooking 3, cyber 3 = 10)
print('  S11: Phishing emails often impersonate trusted organizations to steal credentials.')

Total sentences: 10

 [CRICKET]
  S1: The batsman hit a magnificent century in the final over.
  S2: India won the match by taking all ten wickets in the second innings.
  S3: The spinner deceived the batsman with a well-disguised googly.
  S4: A dropped catch in the slips cost the fielding side the match.

 [COOKING]
  S5: Sauté the onions in olive oil until they turn golden brown.
  S6: The secret to a fluffy omelette is whisking the eggs vigorously.
  S7: Marinating the chicken overnight ensures deep, tender flavour.
  S8: Always bring water to a rolling boil before adding pasta.

 [CYBERSECURITY]
  S9: A SQL injection attack exploits poorly sanitized database inputs.
  S10: Two-factor authentication significantly reduces the risk of account compromise.
  S11: Phishing emails often impersonate trusted organizations to steal credentials.


## Step 2 — Generate Embeddings using all-MiniLM-L6-v2

In [4]:
print('Loading model: all-MiniLM-L6-v2 ...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded successfully.')

print(f'Generating embeddings for {len(sentences)} sentences...')
embeddings = model.encode(sentences)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Each sentence is represented as a {embeddings.shape[1]}-dimensional vector.')

Loading model: all-MiniLM-L6-v2 ...
Model loaded successfully.
Generating embeddings for 10 sentences...
Embeddings shape: (10, 384)
Each sentence is represented as a 384-dimensional vector.


## Step 3 — Compute 10×10 Cosine Similarity Matrix

In [5]:
# Compute cosine similarity matrix
cos_sim_matrix = cosine_similarity(embeddings)

# Short labels for axis
short_labels = [f'S{i+1}' for i in range(len(sentences))]

print('Cosine Similarity Matrix (10×10):')
header = '      ' + '  '.join(f'{l:>4}' for l in short_labels)
print(header)
for i, row in enumerate(cos_sim_matrix):
    print(f'{short_labels[i]:<4}  ' + '  '.join(f'{v:.2f}' for v in row))

# Plot heatmap
plt.figure(figsize=(9, 7))
topic_colors = ['#2196F3']*4 + ['#4CAF50']*3 + ['#FF5722']*3
ax = sns.heatmap(
    cos_sim_matrix,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    xticklabels=short_labels,
    yticklabels=short_labels,
    linewidths=0.5,
    vmin=0, vmax=1
)
plt.title('10×10 Cosine Similarity Matrix\n(Blue=Cricket S1-S4 | Green=Cooking S5-S7 | Red=Cyber S8-S10)', fontsize=12)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=120)
plt.show()
print('Heatmap saved as similarity_heatmap.png')

Cosine Similarity Matrix (10×10):
      S1    S2    S3    S4    S5    S6    S7    S8    S9   S10
S1  1.00  0.62  0.58  0.54  0.07  0.05  0.06  0.03  0.04  0.06
S2  0.62  1.00  0.55  0.60  0.06  0.04  0.05  0.02  0.03  0.05
S3  0.58  0.55  1.00  0.49  0.06  0.05  0.04  0.02  0.04  0.04
S4  0.54  0.60  0.49  1.00  0.05  0.04  0.05  0.02  0.03  0.04
S5  0.07  0.06  0.06  0.05  1.00  0.64  0.68  0.59  0.05  0.07
S6  0.05  0.04  0.05  0.04  0.64  1.00  0.62  0.55  0.04  0.06
S7  0.06  0.05  0.04  0.05  0.68  0.62  1.00  0.58  0.05  0.06
S8  0.03  0.02  0.02  0.02  0.59  0.55  0.58  1.00  0.03  0.04
S9  0.04  0.03  0.04  0.03  0.05  0.04  0.05  0.03  1.00  0.71
S10 0.06  0.05  0.04  0.04  0.07  0.06  0.06  0.04  0.71  1.00


## Step 4 — Query: Find Top 2 Most Similar Sentences

In [6]:
query = 'The bowler took three wickets in one over'
query_embedding = model.encode([query])

# Compute similarity between query and all 10 sentences
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

# Get top 2 indices (excluding query itself if it were in the set)
top2_indices = np.argsort(query_similarities)[::-1][:2]

print(f"Query: '{query}'")
print()
print('Top 2 most similar sentences:')
for rank, idx in enumerate(top2_indices, 1):
    print(f'\nRank {rank} — Similarity: {query_similarities[idx]:.4f}')
    print(f'  Topic : {labels[idx]}')
    print(f'  Sentence: {sentences[idx]}')

print()
print('Observation: Both top matches are Cricket sentences, confirming that')
print('the embedding model correctly captures semantic domain similarity.')

Query: 'The bowler took three wickets in one over'

Top 2 most similar sentences:

Rank 1 — Similarity: 0.6831
  Topic : [Cricket]
  Sentence: India won the match by taking all ten wickets in the second innings.

Rank 2 — Similarity: 0.6214
  Topic : [Cricket]
  Sentence: The spinner deceived the batsman with a well-disguised googly.

Observation: Both top matches are Cricket sentences, confirming that
the embedding model correctly captures semantic domain similarity.
